In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))

# Enrich busyaprime factorization with brot, words1, words2
import numpy as np
bp_npz = bp_members / 'dfthcsos__submission.npz'
npz = np.load(bp_npz, allow_pickle=False)
data = {k: np.array(npz[k]) for k in npz.keys()}
data['brot'] = np.int64(8)

# Compute angle words using encode_inputs (need starter_kit)
try:
    sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
    from starter.task1_factorization.solution_io import read_factorization
    from starter.task2_angles.encode import TREES, encode_inputs
    from verifier.score import load_inputs
    params, _ = read_factorization(bp_npz)
    inputs = load_inputs('FeMoCo76o113e', params)
    tree_edges = np.array(npz['tree_edges'])
    words = encode_inputs(inputs, tree_edges, 8, 'nearest')
    data['angle_words'] = words
    data['words1'] = words[:76]
    data['words2'] = words[76:]
    print('Enriched with angle words')
except Exception as e:
    print('Could not enrich:', e)
    # Try to copy brot from Michael's factorization
    mr_npz = np.load('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/fact/dfthc_r10_b129_c46_brot8.npz', allow_pickle=False)
    data['brot'] = mr_npz['brot']
    data['angle_words'] = mr_npz['angle_words']
    data['words1'] = mr_npz['words1']
    data['words2'] = mr_npz['words2']
    print('Used Michael brot/words')

enriched = WORK / 'enriched.npz'
np.savez(enriched, **data)
shutil.copy2(enriched, sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / 'fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers'))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'fin.npz').exists())


In [ ]:
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / 'fin.npz'), str(RUNS / 'g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers'))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'g.npz').exists())


In [ ]:
# Save the optimized factorization
final = RUNS / 'g.npz'
if not final.exists():
    final = RUNS / 'fin.npz'
print('Using:', final)
shutil.copy2(final, WORK / 'optimized.npz')
shutil.copy2(final, sub_dir / 'dfthcsos' / 'submission.npz')
print('Saved optimized factorization')

# Package (circuits unchanged from busyaprime)
zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
